# 0. Complete SafeDrive Colab driver

This is the single canonical notebook. The final question is whether geometry-competent SAC can adapt to simulator-controlled procedural traffic, reduce collisions, and retain traffic-free generalization. The learned policy controls one ego vehicle only (`num_agents=1`, `is_multi_agent=false`). Run exactly two seed-0 pilots and one selected seed-1 confirmation; do not add seeds or variants automatically.

## 1. Constants and paths

In [1]:
import json
import os
from pathlib import Path

REPO_URL = "https://github.com/djdhillxn/safedrive.git"
REPO_DIR = Path("/content/safedrive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/SafeDrive")
RUNS_DIR = REPO_DIR / "runs"
CPU_COUNT = os.cpu_count() or 2
N_ENVS = min(4, max(1, CPU_COUNT - 1))
VEC_ENV = "subproc" if N_ENVS > 1 else "dummy"
print({"CPU_COUNT": CPU_COUNT, "N_ENVS": N_ENVS, "VEC_ENV": VEC_ENV})

{'CPU_COUNT': 12, 'N_ENVS': 4, 'VEC_ENV': 'subproc'}


## 2. Runtime, CPU, RAM, CUDA, and L4 inspection

In [2]:
import platform
print(platform.platform())
# Inspect native packages in a disposable process before project installation.
!python -c "import os, psutil, torch; print('CPU: %s; RAM: %.1f GiB' % (os.cpu_count() or 2, psutil.virtual_memory().total / 2**30)); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA GPU')"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true

Linux-6.6.122+-x86_64-with-glibc2.35
CPU: 12; RAM: 53.0 GiB
CUDA: True
NVIDIA L4
NVIDIA L4, 23034 MiB


## 3. Google Drive mount

In [3]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Clone or fast-forward pull repository

In [4]:
if not (REPO_DIR / ".git").exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main
    !git -C {REPO_DIR} merge --ff-only origin/main
%cd /content/safedrive

From https://github.com/djdhillxn/safedrive
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/safedrive


## 5. Install repository and verify package versions

In [5]:
import sys
from importlib.metadata import PackageNotFoundError, version as package_version

# A compiled package already imported by this kernel cannot safely be replaced in place.
native_modules = {"numpy": "numpy", "opencv-python": "cv2", "panda3d": "panda3d"}
native_versions_before_install = {}
for package_name in native_modules:
    try:
        native_versions_before_install[package_name] = package_version(package_name)
    except PackageNotFoundError:
        native_versions_before_install[package_name] = "missing"
loaded_native_packages = {}
for package_name, module_name in native_modules.items():
    loaded_module = sys.modules.get(module_name)
    if loaded_module is not None:
        loaded_native_packages[package_name] = {
            "module_version": getattr(loaded_module, "__version__", "unknown"),
            "installed_before": native_versions_before_install[package_name],
        }

!python -m pip uninstall -y gym
if _exit_code != 0:
    raise RuntimeError("Removing the obsolete Gym package failed.")
!python -m pip install --disable-pip-version-check -e .
if _exit_code != 0:
    raise RuntimeError("Project installation failed in the clean runtime.")
# Verify compiled packages in a new process; no native wheel is replaced later.
!python -c "import cv2, gymnasium, metadrive, numpy, panda3d, stable_baselines3, torch; print('MetaDrive', getattr(metadrive, '__version__', 'pinned source'), '| Panda3D', panda3d.__version__, '| NumPy', numpy.__version__, '| OpenCV', cv2.__version__, '| SB3', stable_baselines3.__version__, '| Gymnasium', gymnasium.__version__, '| PyTorch', torch.__version__)"
if _exit_code != 0:
    raise RuntimeError("Installed package verification failed; restart from a clean runtime.")

installed_native_versions = {}
for package_name in native_modules:
    try:
        installed_native_versions[package_name] = package_version(package_name)
    except PackageNotFoundError:
        installed_native_versions[package_name] = "missing"
replaced_loaded_packages = {
    package_name: {
        **loaded_details,
        "installed_after": installed_native_versions[package_name],
    }
    for package_name, loaded_details in loaded_native_packages.items()
    if loaded_details["installed_before"] != installed_native_versions[package_name]
}
if replaced_loaded_packages:
    print("Native packages changed in the live kernel:", replaced_loaded_packages)
    raise RuntimeError(
        "Restart the Colab session now, rerun Sections 1-5, and do not continue to "
        "Section 6 in this process. The second Section 5 run will not reinstall them."
    )

Obtaining file:///content/safedrive
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached metadrive_simulator-0.4.3-py3-none-any.whl
  Building editable for saferl-drive-metadrive-mvp (pyproject.toml) ... done
  Created wheel for saferl-drive-metadrive-mvp: filename=saferl_drive_metadrive_mvp-0.1.0-0.editable-py3-none-any.whl size=9833 sha256=23a5b9afc225e95747197f9e177a30a8a3d53c7ab0e5c41176c91f7b6416359b
  Stored in directory: /tmp/pip-ephem-wheel-cache-q52a6nnq/wheels/3e/98/13/d8f6cc1b539fa59a18850bba2492c6f37b39cbc1d017527fec
Successfully built saferl-drive-metadrive-mvp
  Attempting uninstall: saferl-drive-metadrive-mvp
    Found existing installation: saferl-drive-metadrive-mvp 0.1.0
    Uninstalling saferl-drive-metadrive-mvp-0.1.0:
      Successfully uninstalled saferl-drive-metadrive-mvp-0.1.0
2026-07-24 00:21:

## 6. Restore artifacts from Drive

In [6]:
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --local-runs "{RUNS_DIR}" --include-training-artifacts

def optional_latest(name):
    pointer = RUNS_DIR / f"latest_{name}.txt"
    if not pointer.exists():
        return None
    value = Path(pointer.read_text().strip())
    return value if value.is_absolute() else REPO_DIR / value

def run_status(run_dir):
    if run_dir is None:
        return "missing"
    state = run_dir / "curriculum_state.json"
    metadata = run_dir / "run_metadata.json"
    path = state if state.exists() else metadata
    return json.loads(path.read_text()).get("status", "unknown")

def training_action(pointer_name):
    run_dir = optional_latest(pointer_name)
    status = run_status(run_dir)
    if status == "complete":
        return "reuse", run_dir
    if status == "paused":
        return "resume", run_dir
    if status == "failed_gate":
        return "terminal", run_dir
    if status == "missing":
        return "start", None
    raise RuntimeError(f"Refusing ambiguous run status {status!r}: {run_dir}")

Drive source: /content/drive/MyDrive/SafeDrive/runs
Local destination: /content/safedrive/runs
Sync complete: 0 files updated across 0 run directories.
Training artifacts: included (full Colab restore).
Latest IDM: /content/safedrive/runs/20260722_042836_idm_baseline_seed0
Latest PHASE2_EXPERT: /content/safedrive/runs/20260722_155526_expert_baseline_seed0
Latest PHASE2_IDM: /content/safedrive/runs/20260722_155326_idm_baseline_seed0
Latest PPO: /content/safedrive/runs/20260722_012443_ppo_phase1_control_ppo_seed0
Latest PPO_PILOT: /content/safedrive/runs/20260721_230755_ppo_control_pilot_ppo_seed0
Latest SAC: /content/safedrive/runs/20260722_012638_sac_phase1_control_sac_seed0
Latest SAC_PHASE2_CURRICULUM_SEED0: /content/safedrive/runs/20260722_071416_sac_phase2_curriculum_sac_seed0
Latest SAC_PHASE2_CURRICULUM_SEED1: /content/safedrive/runs/20260722_092850_sac_phase2_curriculum_sac_seed1
Latest SAC_PHASE2_CURRICULUM_SEED2: /content/safedrive/runs/20260722_130907_sac_phase2_curriculum_sa

## 7. Separate training-critical and rendering-capability gates

### 7A. Training-critical gate (blocking)

In [8]:
from importlib.metadata import version
from saferl_drive.config import load_yaml
from scripts.train_curriculum import validate_curriculum_config

required_runtime = {
    "numpy": "2.0.2",
    "opencv-python": "4.11.0.86",
    "panda3d": "1.10.16",
}
runtime_mismatches = {
    package_name: {"required": required, "installed": version(package_name)}
    for package_name, required in required_runtime.items()
    if version(package_name) != required
}
if runtime_mismatches:
    raise RuntimeError(
        f"Training runtime does not match the checkpoint-compatible pins: {runtime_mismatches}. "
        "Start a clean runtime and rerun Sections 1-5."
    )
validate_curriculum_config(load_yaml("configs/sac_traffic_curriculum.yaml"))
!python -m compileall saferl_drive scripts tests
if _exit_code != 0:
    raise RuntimeError("Python compilation failed; stop before running experiments.")
!pytest -q
if _exit_code != 0:
    raise RuntimeError("Repository tests failed; stop before running experiments.")
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split validation --episodes 1 --densities 0.05 --prefix traffic_training_smoke --training-smoke --no-progress validation.start_seed=50000 validation.num_scenarios=1
if _exit_code != 0:
    raise RuntimeError("The training-critical physics/vector/traffic smoke failed.")
SOURCE_CHECKPOINT_RUN = optional_latest("sac_phase2_curriculum_seed0")
if SOURCE_CHECKPOINT_RUN is None:
    raise FileNotFoundError("The seed-0 geometry checkpoint was not restored from Drive.")
!python -m scripts.evaluate --run-dir "{SOURCE_CHECKPOINT_RUN}" --model best --split validation --episodes 1 --densities 0.05 --prefix training_gate_checkpoint --no-progress validation.start_seed=50000 validation.num_scenarios=1
if _exit_code != 0:
    raise RuntimeError(
        "The restored SAC checkpoint gate failed. Read the evaluator traceback above: "
        "it now distinguishes checkpoint deserialization from a genuine SB3 "
        "observation/action-space mismatch."
    )
TRAINING_STATUS = "READY"
print("TRAINING STATUS: READY")
print("Training-critical simulation checks passed. Chase rendering is a presentation-only capability and does not affect the LidarState SAC training environment.")

Listing 'saferl_drive'...
Listing 'scripts'...
Listing 'tests'...
...........................................................              [100%]
59 passed in 7.63s
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Training-critical environment smoke passed: {'observation_shape': [259], 'observation_dtype': 'float32', 'action_shape': [2], 'action_dtype': 'float32', 'num_agents': 1, 'is_multi_agent': False, 'image_observation': False, 'render_mode': 'none', 'traffic_density': 0.05, 'traffic_vehicle_count': 12, 'status': 'passed'}
INFO: Evaluating ExpertPolicy for 1 episodes at traffic density 0.05.
INFO: ExpertPolicy baseline matrix complete: runs/20260724_002456_expert_traffic_baseline_seed0
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
ERROR: Evaluation failed.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/safedrive/scripts/evaluate.py", li

RuntimeError: The restored SAC checkpoint is incompatible with the target vector/action interface.

### 7B. Rendering-capability gate (diagnostic and non-blocking)

In [ ]:
import os
import platform
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from scripts.record_video import make_rendering_status

if TRAINING_STATUS != "READY":
    raise RuntimeError("Run the training-critical gate before diagnosing rendering.")
RENDER_STATUS_PATH = RUNS_DIR / "rendering_status.json"
official_log = RUNS_DIR / "metadrive_headless_verifier.log"
bare_log = RUNS_DIR / "safedrive_bare_renderer_smoke.log"
full_log = RUNS_DIR / "safedrive_chase_mp4_smoke.log"
bare_directory = Path("/tmp/safedrive_renderer_smoke")
chase_video = Path("/tmp/safedrive_chase_smoke.mp4")
chase_sidecar = chase_video.with_suffix(".json")
metadrive_spec = find_spec("metadrive")
if metadrive_spec is None or not metadrive_spec.submodule_search_locations:
    raise RuntimeError("MetaDrive is not installed in this runtime.")
metadrive_package_dir = Path(next(iter(metadrive_spec.submodule_search_locations)))
expected_official_pngs = [
    metadrive_package_dir / "examples" / "main_camera_from_observation.png",
    metadrive_package_dir / "examples" / "main_camera_from_buffer.png",
]
for path in expected_official_pngs:
    path.unlink(missing_ok=True)
for path in bare_directory.glob("main_camera_*.png") if bare_directory.exists() else []:
    path.unlink()
(bare_directory / "renderer_smoke.json").unlink(missing_ok=True)
chase_video.unlink(missing_ok=True)
chase_sidecar.unlink(missing_ok=True)

package_versions = {}
for package in ["metadrive-simulator", "panda3d", "numpy", "opencv-python"]:
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = None
diagnostics = {
    "python": platform.python_version(),
    "packages": package_versions,
    "metadrive_pinned_commit": "85e5dadc6c7436d324348f6e3d8f8e680c06b4db",
    "DISPLAY": os.environ.get("DISPLAY"),
    "WAYLAND_DISPLAY": os.environ.get("WAYLAND_DISPLAY"),
    "official_log": str(official_log),
    "bare_log": str(bare_log),
    "full_log": str(full_log),
}
boundaries = []
first_failure = None
failure_exit = None

!bash -o pipefail -c "python -m metadrive.examples.verify_headless_installation --camera main 2>&1 | tee '{official_log}'"
official_exit = int(_exit_code)
official_pngs = {path.name: path.exists() and path.stat().st_size > 0 for path in expected_official_pngs}
boundaries.append({"name": "official_metadrive_verifier", "exit_code": official_exit, "pngs": official_pngs})
if official_exit != 0 or not all(official_pngs.values()):
    first_failure = "official_metadrive_verifier"
    failure_exit = official_exit
pipe_types_log = RUNS_DIR / "rendering_graphics_pipe_types.txt"
!python -c "from panda3d.core import GraphicsPipeSelection; selection = GraphicsPipeSelection.get_global_ptr(); selection.load_aux_modules(); print('\\n'.join(str(pipe_type) for pipe_type in selection.get_pipe_types()))" > "{pipe_types_log}" 2>&1
diagnostics["graphics_pipe_query_exit_code"] = int(_exit_code)
diagnostics["known_graphics_pipe_types"] = pipe_types_log.read_text().splitlines() if pipe_types_log.exists() else []

if first_failure is None:
    !bash -o pipefail -c "python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --smoke-renderer --view chase --density 0.05 --seed 50000 --steps 5 --width 320 --height 192 --output /tmp/safedrive_renderer_smoke 2>&1 | tee '{bare_log}'"
    bare_exit = int(_exit_code)
    bare_summary = bare_directory / "renderer_smoke.json"
    bare_pngs = list(bare_directory.glob("main_camera_*.png"))
    bare_ok = bare_exit == 0 and bare_summary.exists() and len(bare_pngs) >= 3
    boundaries.append({"name": "safedrive_bare_camera", "exit_code": bare_exit, "frame_count": len(bare_pngs)})
    if not bare_ok:
        first_failure = "safedrive_bare_camera"
        failure_exit = bare_exit

if first_failure is None:
    !bash -o pipefail -c "python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed 50000 --steps 5 --width 320 --height 192 --output /tmp/safedrive_chase_smoke.mp4 2>&1 | tee '{full_log}'"
    full_exit = int(_exit_code)
    full_text = full_log.read_text() if full_log.exists() else ""
    frame_collection_passed = "frame-collection boundary passed" in full_text
    full_ok = full_exit == 0 and chase_video.exists() and chase_video.stat().st_size > 0 and chase_sidecar.exists()
    boundaries.append({"name": "safedrive_sb3_recorder", "exit_code": full_exit, "frame_collection_passed": frame_collection_passed})
    if full_ok:
        boundaries.append({"name": "imageio_mp4_encoding", "exit_code": 0, "video_bytes": chase_video.stat().st_size, "sidecar": True})
    if not full_ok:
        first_failure = "imageio_mp4_encoding" if frame_collection_passed else "safedrive_sb3_recorder"
        failure_exit = full_exit

diagnostics["boundaries"] = boundaries
diagnostics["official_pngs"] = official_pngs
if first_failure is None:
    rendering_record = make_rendering_status("passed", "imageio_mp4_encoding", diagnostics=diagnostics)
    RENDERING_STATUS = "PASSED"
else:
    nvidia_log = RUNS_DIR / "rendering_nvidia.txt"
    libraries_log = RUNS_DIR / "rendering_graphics_libraries.txt"
    !nvidia-smi > "{nvidia_log}" 2>&1 || true
    !bash -c "ldconfig -p 2>/dev/null | grep -E 'lib(EGL|GLX|GL)\\.so' > '{libraries_log}' || true"
    diagnostics["nvidia_log"] = str(nvidia_log)
    diagnostics["graphics_libraries_log"] = str(libraries_log)
    rendering_record = make_rendering_status("failed", first_failure, exit_code=failure_exit, diagnostics=diagnostics)
    RENDERING_STATUS = "FAILED"
RENDER_STATUS_PATH.write_text(json.dumps(rendering_record, indent=2) + "\n")
print(f"RENDERING STATUS: {RENDERING_STATUS}")
if RENDERING_STATUS == "FAILED":
    print(f"WARNING: chase rendering stopped at {first_failure}; diagnostics were saved to {RENDER_STATUS_PATH}.")
    print("TRAINING REMAINS AUTHORIZED: SAC uses vector LidarState observations and no camera.")
else:
    print("Official MetaDrive, direct SafeDrive camera, and complete MP4 gates all passed.")

## 8. Summarize existing Phase-1 and Phase-2 results without retraining

In [ ]:
import pandas as pd
for path in [RUNS_DIR / "phase1_comparison.csv", RUNS_DIR / "phase2_comparison.csv", RUNS_DIR / "phase2_light_traffic_comparison.csv"]:
    if path.exists():
        display(pd.read_csv(path))
print("Historical Phase 1/2 runs are evidence and are never retrained by this notebook.")

## 9. Experiment 0: IDM and ExpertPolicy traffic solvability

In [ ]:
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy idm --split validation --episodes 50 --densities 0.05 0.10 --prefix traffic_solvability_idm validation.num_scenarios=50
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split validation --episodes 50 --densities 0.05 0.10 --prefix traffic_solvability_expert validation.num_scenarios=50
if RENDERING_STATUS == "PASSED":
    !python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed 50000 --steps 250 --output runs/videos/expert_solubility_chase_d005_seed50000.mp4
else:
    print("Skipping the presentation-only ExpertPolicy video because Section 7B did not establish renderer capability.")
for name in ["traffic_extension_idm", "traffic_extension_expert"]:
    baseline_run = optional_latest(name)
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{baseline_run}"

## 10. Experiment 1: frozen pre-adaptation evaluation matrix

In [ ]:
for seed in [0, 1]:
    source_run = optional_latest(f"sac_phase2_curriculum_seed{seed}")
    if source_run is None:
        raise FileNotFoundError(f"Restore the completed curriculum seed-{seed} run from Drive.")
    !python -m scripts.evaluate --run-dir "{source_run}" --model best --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_before
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{source_run}"
SOURCE_SEED0_RUN = optional_latest("sac_phase2_curriculum_seed0")
!python -m scripts.evaluate --run-dir "{SOURCE_SEED0_RUN}" --model best --split validation --episodes 50 --densities 0.0 0.05 0.10 --prefix traffic_source_validation validation.num_scenarios=50
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{SOURCE_SEED0_RUN}"

## 11. Experiment 2: seed-0 SAC-Traffic pilot

In [ ]:
action, REFERENCE_RUN = training_action("sac_traffic_reference_seed0")
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant reference --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant reference --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{REFERENCE_RUN}"
else:
    print(f"Reference pilot action: {action}; run: {REFERENCE_RUN}")
REFERENCE_RUN = optional_latest("sac_traffic_reference_seed0")
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{REFERENCE_RUN}"

## 12. Experiment 3: seed-0 SAC-Traffic-Safe pilot

In [ ]:
action, SAFETY_RUN = training_action("sac_traffic_safety_seed0")
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant safety --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant safety --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{SAFETY_RUN}"
else:
    print(f"Safety pilot action: {action}; run: {SAFETY_RUN}")
SAFETY_RUN = optional_latest("sac_traffic_safety_seed0")
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{SAFETY_RUN}"

## 13. Pilot comparison and saved selection decision

In [ ]:
for pilot_run in [REFERENCE_RUN, SAFETY_RUN]:
    pilot_status = run_status(pilot_run)
    if pilot_status not in {"complete", "failed_gate"}:
        raise RuntimeError(f"Pilot is not terminal: {pilot_status!r} ({pilot_run})")
    if pilot_status == "failed_gate":
        print(f"Evaluating preserved Stage-A failure checkpoint without treating it as completed adaptation: {pilot_run}")
    !python -m scripts.evaluate --run-dir "{pilot_run}" --model best --split validation --episodes 50 --densities 0.0 0.05 0.10 --prefix traffic_pilot validation.num_scenarios=50
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{pilot_run}"
!python -m scripts.compare_runs --traffic-extension --select-pilots
SELECTION = json.loads((RUNS_DIR / "traffic_extension_selection.json").read_text())
print(json.dumps(SELECTION, indent=2))

## 14. Experiment 5: selected seed-1 confirmation

In [ ]:
SELECTED_VARIANT = SELECTION["selected_variant"]
pointer = f"sac_traffic_{SELECTED_VARIANT}_seed1"
action, CONFIRMATION_RUN = training_action(pointer)
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant {SELECTED_VARIANT} --seed 1 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant {SELECTED_VARIANT} --seed 1 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{CONFIRMATION_RUN}"
else:
    print(f"Confirmation action: {action}; run: {CONFIRMATION_RUN}")
CONFIRMATION_RUN = optional_latest(pointer)
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{CONFIRMATION_RUN}"

## 15. Final evaluation matrix

In [ ]:
confirmation_status = run_status(CONFIRMATION_RUN)
if confirmation_status not in {"complete", "failed_gate"}:
    raise RuntimeError(f"The seed-1 confirmation is not terminal: {confirmation_status!r}.")
if confirmation_status == "failed_gate":
    print("Seed 1 failed its Stage-A gate. Its saved best diagnostic checkpoint will be evaluated, but it will be excluded from completed-lineage aggregates.")
SELECTED_SEED0_RUN = optional_latest(f"sac_traffic_{SELECTED_VARIANT}_seed0")
for adapted_run in [SELECTED_SEED0_RUN, CONFIRMATION_RUN]:
    !python -m scripts.evaluate --run-dir "{adapted_run}" --model best --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{adapted_run}"
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy idm --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final_idm
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final_expert
for name in ["traffic_extension_idm", "traffic_extension_expert"]:
    baseline_run = optional_latest(name)
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{baseline_run}"

## 16. Third-person videos

In [ ]:
if RENDERING_STATUS != "PASSED":
    print("Skipping presentation videos because Section 7B did not establish renderer capability. Training and evaluation results remain valid.")
else:
    from scripts.record_video import select_video_scenario
    ADAPTED_CSV = SELECTED_SEED0_RUN / "eval" / "traffic_final_d005_episodes.csv"
    MATCHED_SEED = select_video_scenario(ADAPTED_CSV, "first")
    SOURCE_RUN = optional_latest("sac_phase2_curriculum_seed0")
    !python -m scripts.record_video --run-dir "{SOURCE_RUN}" --model best --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/frozen_before_matched_chase.mp4
    !python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/adapted_after_matched_chase.mp4
    adapted_frame = pd.read_csv(ADAPTED_CSV)
    if adapted_frame["success"].astype(bool).any():
        !python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --episodes-csv "{ADAPTED_CSV}" --scenario-rule first_success --output runs/videos/adapted_representative_success_chase.mp4
    else:
        print("No successful selected-policy episode exists; no success video will be fabricated.")
    if (~adapted_frame["success"].astype(bool)).any():
        !python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --episodes-csv "{ADAPTED_CSV}" --scenario-rule first_failure --output runs/videos/adapted_diagnostic_failure_chase.mp4
    !python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/expert_demonstration_chase.mp4

## 17. Final plots and tables

In [ ]:
!python -m scripts.compare_runs --traffic-extension
display(pd.read_csv(RUNS_DIR / "traffic_extension_comparison.csv"))

## 18. Build main and surrogate reports

In [ ]:
!latexmk -cd -pdf -interaction=nonstopmode -halt-on-error reports/main.tex
!latexmk -cd -pdf -interaction=nonstopmode -halt-on-error reports/surrogate_notes.tex

## 19. Final Drive synchronization

In [ ]:
for run_dir in [SELECTED_SEED0_RUN, CONFIRMATION_RUN]:
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{run_dir}"
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --project-artifacts-to-drive
print("Final artifacts persisted to", DRIVE_PROJECT)

In [ ]:
# End of the canonical notebook.